In [0]:
pip install -q -r requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks_langchain import DatabricksEmbeddings
from langchain_community.document_loaders import JSONLoader
from langchain_chroma import Chroma
from databricks_langchain import ChatDatabricks

In [0]:
loader = JSONLoader(
    file_path="/Workspace/Users/geethu1109@gmail.com/AI_learning/INPUT/sample_error.json", jq_schema=".[]",
    text_content=False
)
 
documents = loader.load()
print(len(documents))

In [0]:
embeddings = DatabricksEmbeddings(
    endpoint="databricks-bge-large-en"
)

In [0]:
vectordb = Chroma.from_documents(documents, embeddings, persist_directory="./chroma_index")

In [0]:
# query = """ailed\\nAttempting retries: none\\n----\\n##[error]AADSTS7000222: client secret expired\\n##[error]Process completed with exit code 1\\nCleaning up workspace", "solution": "Rotate the SPN client secret in Azure AD, copy the **secret value**, update the GitHub Actions secret, and rerun the workflow. Consider monitoring secret expiry date"""
query="glapi"
results = vectordb.similarity_search_with_score(query, k=4)
filter_results = []
for i in results:
    if i[1] < 0.8:
        filter_results.append(i)
print('Matching solutions :',len(filter_results))
if not filter_results:
    print("No sufficiently relevant solution found (score >= 0.8).")

In [0]:
candidates = []

for idx, (doc, score) in enumerate(filter_results, start=1):
    candidates.append(f"""
Candidate {idx}:
Problem:
{doc.page_content}

Solution:
{doc.metadata.get('solution', 'No solution provided')}

Similarity score: {score}
""")

candidates_text = "\n".join(candidates)
# print(candidates_text)

In [0]:
llm_prompt = f"""
You are an expert assistant specialized in analyzing CI/CD build failure logs.

INPUT:
------
Build Error Log:
{query}

Candidate Solutions (retrieved from a known knowledge base):
{candidates_text}

TASK:
-----
1. Carefully analyze whether the build error log matches any of the candidate solutions.
2. Select the ONE solution that most directly and accurately resolves the error.

DECISION RULES:
---------------
- If exactly one candidate solution clearly applies, return ONLY that solution text.
- If NONE of the candidate solutions apply, classify the result as: NO_MATCH.   
NO_MATCH HANDLING:
-----------------
If the result is NO_MATCH, first determine whether the Build Error Log resembles CI/CD build or deployment errors.
- First determine whether the Build Error Log text is related to CI/CD build or deployment errors.

a) IF the Build Error Log is NOT related to CI/CD build or deployment logs:
   - Respond using EXACTLY two sentences, in the order shown below:
     1. A polite greeting stating that you are an AI assistant designed to analyze CI/CD build error logs and requesting the user to share relevant build or pipeline error logs.
     2. A short statement saying that the provided input does not resemble a CI/CD build or deployment error log.
   - Do NOT include any technical analysis, reasoning, or diagnostic phrasing.
   - Do NOT change the sentence order.

b) IF the Build Error Log IS related to CI/CD build logs:
   - Provide GENERAL, NON-SPECIFIC troubleshooting steps relevant to the error.
   - Do NOT reference any candidate solutions.
   - Do NOT assume unavailable context.
   - Present the steps as a numbered list.
   - Limit the response to 4–5 concise lines.



STRICT CONSTRAINTS:
-------------------
- Do NOT invent or fabricate a specific fix when a matching solution exists.
- Do NOT output explanations, reasoning, headings, or extra commentary.
- Follow the output rules exactly.
"""


In [0]:
llm = ChatDatabricks(
    model="databricks-llama-4-maverick",
    temperature=0
)


In [0]:

best_solution = llm.invoke(llm_prompt)

print(best_solution.content)